# Weitere Metriken

Tabelle 8 und Anhang E.1, ohne zusätzliche Grafik. ROUGE-1/2/L und BERTScore verwenden die CNN/DailyMail-Highlights als Referenz; SummaC entweder den Originalartikel oder den unmittelbaren Stufeneingang. Bei leerem Stufeneingang ist der lokale Score analytisch fehlend. Modellwerte werden gespeichert übernommen; keine erneute Modellbewertung.

In [1]:
from pathlib import Path
from fractions import Fraction
from collections import Counter
from itertools import combinations
import hashlib, importlib.metadata, json, math, platform, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display
from threadpoolctl import threadpool_limits
from faktenretention.io import PROJECT_ROOT
from faktenretention.data import load_inputs, panel, length_values, reconstruct_qa, aggregate_articles, CELL, KEY
from faktenretention.protocol import METHODS, STAGES, WINDOWS, CONTRASTS, LABELS
threadpool_limits(limits=1)
SEED, B, N_FACTS = (32, 10000, 20)
DATA_DIR = PROJECT_ROOT / 'data'
OUT = PROJECT_ROOT / 'results/05_weitere_metriken'
FIG = OUT
OUT.mkdir(parents=True, exist_ok=True)
COLORS = dict(zip(METHODS, ['#777777', '#0072B2', '#E69F00', '#009E73', '#CC79A7', '#D55E00']))
MARKERS = dict(zip(METHODS, ['o', 's', '^', 'D', 'v', 'P']))
LINESTYLES = dict(zip(METHODS, ['--', '-', '-.', '--', ':', '-']))
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.max_rows', 26)

def export(frame, name):
    frame.to_csv(OUT / f'{name}.csv', index=False)

def fmt(x, d=2):
    return f'{x:.{d}f}'.replace('.', ',')

def savefig(fig, name):
    fig.savefig(FIG / f'{name}.png', dpi=220, bbox_inches='tight')
    plt.show()
    plt.close(fig)

In [2]:
inputs = load_inputs()
qa, article = panel(inputs)
facts = inputs['facts']
summaries = inputs['summaries']
articles = inputs['articles']
IDS = sorted(articles.article_id)
assert len(IDS) == 100

In [3]:
from scipy.stats import kendalltau
NAMES = LABELS
SECONDARY = ['highlight_rougeL_f1', 'highlight_bertscore_f1', 'source_summac', 'stage_input_summac']
SCORES = ['highlight_rouge1_f1', 'highlight_rouge2_f1'] + SECONDARY
METRICS = ['R_obs', 'R_adj', 'W'] + SCORES
MET_NAMES = {'R_obs': 'QA-Retention', 'R_adj': 'Adjustierte Retention', 'W': 'Wortzahl', 'highlight_rouge1_f1': 'ROUGE-1 F1',
    'highlight_rouge2_f1': 'ROUGE-2 F1', 'highlight_rougeL_f1': 'ROUGE-L F1', 'highlight_bertscore_f1': 'BERTScore F1',
    'source_summac': 'SummaC: Original', 'stage_input_summac': 'SummaC: Stufeneingang'}
raw_scores = inputs['metrics']
analysis = article.merge(raw_scores[CELL + SCORES + ['stage_actual_words']], on=CELL, validate='one_to_one')
assert analysis.W.eq(analysis.stage_actual_words).all()
assert analysis.loc[analysis.empty_output, SCORES].eq(0).all().all()
assert analysis.loc[analysis.stage_target_words.eq(500), 'source_summac'].eq(analysis.loc[analysis.stage_target_words.eq(500),
    'stage_input_summac']).all()
analysis.loc[analysis.empty_input, 'stage_input_summac'] = np.nan

## Gruppenwerte und Tabelle 8

Mittelwerte und punktweise 95-%-Perzentilintervalle aus 10.000 Artikelbootstrap-Ziehungen (Seed 32).

In [4]:
rng = np.random.default_rng(SEED)
indices = rng.integers(0, len(IDS), size=(B, len(IDS)))
weights = np.zeros((B, len(IDS)), dtype=np.int16)
np.add.at(weights, (np.repeat(np.arange(B), len(IDS)), indices.ravel()), 1)

def estimates(values, ids=IDS, draw_weights=weights):
    a = pd.Series(values, index=ids, dtype=float).reindex(ids).to_numpy()
    valid = np.isfinite(a)
    if not valid.any():
        return np.full(len(draw_weights), np.nan)
    denom = draw_weights @ valid.astype(float)
    return np.divide(draw_weights @ np.nan_to_num(a), denom, out=np.full(len(draw_weights), np.nan), where=denom > 0)

def describe(values, ids=IDS, draw_weights=weights):
    a = pd.Series(values, index=ids, dtype=float).to_numpy()
    boot = estimates(a, ids, draw_weights)
    low, high = np.nanquantile(boot, [0.025, 0.975]) if np.isfinite(boot).any() else (np.nan, np.nan)
    valid = a[np.isfinite(a)]
    return (dict(mean=float(valid.mean()) if len(valid) else np.nan, sd=float(valid.std(ddof=1)) if len(valid) > 1 else np.nan,
        n_valid=len(valid), n_missing=len(a) - len(valid), ci_low=low, ci_high=high), boot)
group_rows = []
group_draws = {}
for m in METHODS:
    for t in STAGES:
        g = analysis[analysis.method.eq(m) & analysis.stage_target_words.eq(t)].set_index('article_id').reindex(IDS)
        for metric in METRICS:
            stats, boot = describe(g[metric].to_numpy())
            group_rows.append(dict(method=m, method_name=NAMES[m], stage_target_words=t, metric=metric,
                **stats, n_qa_valid=int(g.n_valid.sum()), n_qa_missing=int(g.n_missing.sum()), n_empty_outputs=int(g.empty_output.sum()),
                n_empty_inputs=int(g.empty_input.sum()), missing_reason='empty_immediate_input' if stats['n_missing'] else 'none'))
            group_draws[f'{m}__{t}__{metric}'] = boot
groups = pd.DataFrame(group_rows)
export(groups, 'mittelwerte_24_gruppen_alle_metriken_95ci')
pass
wide = groups.pivot(index=['method', 'stage_target_words'], columns='metric', values='mean').reset_index()

In [5]:
table8 = wide[wide.stage_target_words.eq(75)].copy()
export(table8, 'tabelle_8_75_woerter')
display(table8.round(4))

metric,method,stage_target_words,R_adj,R_obs,W,highlight_bertscore_f1,highlight_rouge1_f1,highlight_rouge2_f1,highlight_rougeL_f1,source_summac,stage_input_summac
0,auto_cot,75,0.2688,0.3908,115.08,0.9110,0.3174,0.0885,0.1777,0.2627,0.2154
4,automate_cot,75,0.2544,0.3699,115.15,0.9115,0.3148,0.0821,0.1739,0.2641,0.2148
8,few_shot_cot,75,0.2613,0.3455,104.16,0.9121,0.3071,0.0806,0.1737,0.2642,0.2139
12,lead_k,75,0.1730,0.1731,75.00,0.9254,0.3727,0.1492,0.2253,0.2776,0.2178
16,plain,75,0.2472,0.3380,105.70,0.9138,0.3130,0.0856,0.1786,0.2631,0.2184
20,zero_shot_cot,75,0.2634,0.2893,80.35,0.9090,0.3085,0.0816,0.1827,0.2620,0.2147


## Kendall-τ-b je Artikel

Je Artikel/Stufe werden QA-Retention und Sekundärmetrik über alle sechs Verfahren verglichen. Unvollständige oder konstante Vektoren bleiben undefiniert. Mittelwert und 95-%-Artikelbootstrap-Intervall verwenden die gültigen Artikel derselben Ziehung; Gleichstände werden mit τ-b berücksichtigt.

In [6]:
def article_correlations(frame, ids):
    rows = []
    for a in ids:
        for t in STAGES:
            g = frame[frame.article_id.eq(a) & frame.stage_target_words.eq(t)].set_index('method').reindex(METHODS)
            for metric in SECONDARY:
                x = g.R_obs.to_numpy(float)
                y = g[metric].to_numpy(float)
                valid = np.isfinite(x) & np.isfinite(y)
                qt = sum((x[i] == x[j] for i, j in combinations(np.where(valid)[0], 2)))
                mt = sum((y[i] == y[j] for i, j in combinations(np.where(valid)[0], 2)))
                if valid.sum() < 6:
                    tau = np.nan
                    reason = 'incomplete_six_method_panel'
                elif len(np.unique(x)) == 1:
                    tau = np.nan
                    reason = 'all_qa_tied'
                elif len(np.unique(y)) == 1:
                    tau = np.nan
                    reason = 'all_secondary_tied'
                else:
                    tau = float(kendalltau(x, y, variant='b').statistic)
                    reason = '' if np.isfinite(tau) else 'undefined_other'
                rows.append(dict(article_id=a, stage_target_words=t, metric=metric, tau_b=tau, n_methods_valid=int(valid.sum()),
                    qa_tie_pairs=qt, secondary_tie_pairs=mt, undefined_reason=reason))
    return pd.DataFrame(rows)
article_tau = article_correlations(analysis, IDS)
export(article_tau, 'kendall_tau_b_artikelebene_1600')
tau_rows = []
tau_draws = {}
for t in STAGES:
    for metric in SECONDARY:
        g = article_tau[article_tau.stage_target_words.eq(t) & article_tau.metric.eq(metric)].set_index('article_id').reindex(IDS)
        stats, boot = describe(g.tau_b.to_numpy())
        tau_rows.append(dict(stage_target_words=t, metric=metric, **stats, n_articles_with_qa_ties=int(g.qa_tie_pairs.gt(0).sum()),
            undefined_reasons=json.dumps(g.loc[g.tau_b.isna(), 'undefined_reason'].value_counts().to_dict())))
        tau_draws[f'{t}__{metric}'] = boot
tau_summary = pd.DataFrame(tau_rows)
export(tau_summary.assign(metric_name=tau_summary.metric.map(MET_NAMES)), 'anhang_E_1_kendall_tau_b')
pass
use_tau = tau_summary.n_valid.ge(80).all()
display(tau_summary[['stage_target_words', 'metric', 'mean', 'ci_low', 'ci_high', 'n_valid', 'n_missing']].round(3))

,stage_target_words,metric,mean,ci_low,ci_high,n_valid,n_missing
0,500,highlight_rougeL_f1,-0.346,-0.410,-0.278,100,0
1,500,highlight_bertscore_f1,-0.143,-0.217,-0.070,100,0
2,500,source_summac,0.067,-0.004,0.139,100,0
3,500,stage_input_summac,0.067,-0.004,0.139,100,0
4,300,highlight_rougeL_f1,-0.327,-0.383,-0.269,100,0
5,300,highlight_bertscore_f1,-0.315,-0.376,-0.254,100,0
6,300,source_summac,0.004,-0.072,0.080,100,0
7,300,stage_input_summac,-0.053,-0.120,0.014,100,0
8,150,highlight_rougeL_f1,-0.335,-0.397,-0.272,100,0
9,150,highlight_bertscore_f1,-0.544,-0.593,-0.494,100,0
